In [1]:
!pip install -q gradio requests

In [2]:
import gradio as gr
import requests
from datetime import datetime

# =========================
# 설정 영역
# =========================

# 현재는 Colab에서 버튼 동작만 확인하는 실습 모드입니다.
# False = 화면에서 명령만 출력
# True  = 젯슨 오린 나노 서버로 명령 전송
SEND_TO_JETSON = False  # ★ 젯슨 연결 시 True로 변경

# ★ 젯슨 연결 시 수정:
# 젯슨 오린 나노에서 FastAPI 또는 Flask 서버를 실행한 뒤,
# 아래 주소를 젯슨 IP 주소로 변경합니다.
# 예: "http://192.168.0.25:8000"
JETSON_URL = "http://젯슨_IP주소:8000"


# =========================
# 명령 전송 함수
# =========================

def send_command(command, log_text):
    """
    command:
        forward = 전진
        backward = 후진
        left = 좌회전
        right = 우회전
        stop = 정지
    """

    now = datetime.now().strftime("%H:%M:%S")

    # 화면에 보여줄 한글 명령명
    command_name = {
        "forward": "전진",
        "backward": "후진",
        "left": "좌회전",
        "right": "우회전",
        "stop": "정지"
    }

    kor_cmd = command_name.get(command, command)

    # =========================
    # 1) 실습 모드
    # =========================
    if not SEND_TO_JETSON:
        message = f"[{now}] 실습 모드: {kor_cmd} 명령 버튼 클릭"
        new_log = message + "\n" + log_text
        return f"현재 명령: {kor_cmd}", new_log

    # =========================
    # 2) 젯슨 연결 모드
    # =========================
    # ★ 젯슨 연결 시 이 부분이 실제로 동작합니다.
    # 젯슨 오린 나노 쪽에는 아래 주소를 받을 수 있는 서버가 있어야 합니다.
    #
    # 예:
    # POST http://젯슨_IP주소:8000/move/forward
    # POST http://젯슨_IP주소:8000/move/stop

    try:
        response = requests.post(
            f"{JETSON_URL}/move/{command}",
            timeout=2
        )

        message = f"[{now}] 젯슨 전송 성공: {kor_cmd} / 응답: {response.text}"
        new_log = message + "\n" + log_text
        return f"현재 명령: {kor_cmd}", new_log

    except Exception as e:
        message = f"[{now}] 젯슨 연결 실패: {kor_cmd} / 오류: {e}"
        new_log = message + "\n" + log_text
        return "젯슨 연결 실패", new_log


# =========================
# Gradio 버튼 UI
# =========================

with gr.Blocks(title="젯슨 오린 나노 자동차 제어 UI") as demo:
    gr.Markdown("""
    # 젯슨 오린 나노 자동차 제어 버튼 실습

    현재 코드는 Colab에서 버튼 인터페이스를 먼저 연습하는 버전입니다.
    나중에 `SEND_TO_JETSON = True`로 바꾸면 젯슨 오린 나노 서버로 명령을 보낼 수 있습니다.
    """)

    status = gr.Textbox(
        label="현재 상태",
        value="대기 중",
        interactive=False
    )

    log = gr.Textbox(
        label="명령 로그",
        value="",
        lines=10,
        interactive=False
    )

    with gr.Row():
        btn_forward = gr.Button("▲ 전진", variant="primary")

    with gr.Row():
        btn_left = gr.Button("◀ 좌회전")
        btn_stop = gr.Button("■ 정지", variant="stop")
        btn_right = gr.Button("우회전 ▶")

    with gr.Row():
        btn_backward = gr.Button("▼ 후진")

    # 버튼 클릭 이벤트 연결
    btn_forward.click(
        fn=lambda log_text: send_command("forward", log_text),
        inputs=log,
        outputs=[status, log]
    )

    btn_backward.click(
        fn=lambda log_text: send_command("backward", log_text),
        inputs=log,
        outputs=[status, log]
    )

    btn_left.click(
        fn=lambda log_text: send_command("left", log_text),
        inputs=log,
        outputs=[status, log]
    )

    btn_right.click(
        fn=lambda log_text: send_command("right", log_text),
        inputs=log,
        outputs=[status, log]
    )

    btn_stop.click(
        fn=lambda log_text: send_command("stop", log_text),
        inputs=log,
        outputs=[status, log]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://00130b481fed89a732.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
